# Задание 3. Расчет коэффициента D Жуйана

Коэффициент Жуйана показывает, насколько равномерно слово распределено по сегментам корпуса:

$$D = 1 - \frac{\sigma}{\mu \sqrt{n-1}},$$

где $\mu$ — средняя частота слова в сегменте, $\sigma$ — среднее квадратическое отклонение частоты по сегментам, $n$ — число сегментов.

Корпус разбивается на 4 равных сегмента по 1,25 млн словоупотреблений. В каждом сегменте подсчитываются частоты слов из топ-1000 соответствующего словаря. Результат оформляется в виде таблицы: слово, относительная частота, ранг, $D$. Для отчёта берутся 10 самых частых слов каждого словаря.


In [1]:
from pathlib import Path
import json
import re
import time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'freq_ru.json').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'rgr2'
DATA = BASE / 'data'
FIG = BASE / 'figures'
DATA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

DICTIONARIES = [
    ('ru_raw',   'Русский, без лемматизации',  'freq_ru.json',       'ru', 'raw'),
    ('ru_lemma', 'Русский, с лемматизацией',   'freq_ru_lemma.json', 'ru', 'lemma'),
    ('en_raw',   'Английский, без лемматизации', 'freq_en.json',     'en', 'raw'),
    ('en_lemma', 'Английский, с лемматизацией',  'freq_en_lemma.json', 'en', 'lemma'),
]

TARGET_TOKENS = 5_000_000
N_SEGMENTS = 4
SEGMENT_SIZE = TARGET_TOKENS // N_SEGMENTS
TOP_N = 1000

PATTERNS = {
    'ru': re.compile(r'[а-яё]+', re.IGNORECASE),
    'en': re.compile(r'[a-z]+', re.IGNORECASE),
}

def load_dictionary(filename):
    with open(ROOT / filename, encoding='utf-8') as f:
        package = json.load(f)
    items = sorted(package['dictionary'].items(), key=lambda p: (-p[1], p[0]))
    df = pd.DataFrame(items, columns=['word', 'freq'])
    df['rank'] = np.arange(1, len(df) + 1)
    return package, df

packages = {}
datasets = {}
for key, label, filename, lang, variant in DICTIONARIES:
    package, df = load_dictionary(filename)
    packages[key] = package
    datasets[key] = {'df': df, 'label': label, 'lang': lang, 'variant': variant, 'file': filename}
    print(f"{label}: токенов {package['total_tokens']:,}, топ-{TOP_N} подготовлен")


Русский, без лемматизации: токенов 4,982,552, топ-1000 подготовлен


Русский, с лемматизацией: токенов 5,000,251, топ-1000 подготовлен


Английский, без лемматизации: токенов 5,002,249, топ-1000 подготовлен


Английский, с лемматизацией: токенов 5,002,249, топ-1000 подготовлен


In [2]:
targets = {'ru': {'raw': set(), 'lemma': set()}, 'en': {'raw': set(), 'lemma': set()}}
for key, label, filename, lang, variant in DICTIONARIES:
    targets[lang][variant] = set(datasets[key]['df'].head(TOP_N)['word'])
    print(f"{label}: целевых слов {len(targets[lang][variant])}")


Русский, без лемматизации: целевых слов 1000
Русский, с лемматизацией: целевых слов 1000
Английский, без лемматизации: целевых слов 1000
Английский, с лемматизацией: целевых слов 1000


In [3]:
def build_segment_counts(lang):
    """Считает частоты целевых слов в 4 сегментах корпуса и кэширует результат."""
    cache_path = DATA / f'segments_{lang}.json'
    version = 1

    if cache_path.exists():
        with open(cache_path, encoding='utf-8') as f:
            cached = json.load(f)
        if cached.get('version') == version and cached.get('target_tokens') == TARGET_TOKENS:
            print(f'{lang}: загружен кэш {cache_path.name} '
                  f"(токенов {cached['actual_tokens']:,})")
            return cached

    from datasets import load_dataset
    import simplemma

    target_raw = targets[lang]['raw']
    target_lemma = targets[lang]['lemma']
    pattern = PATTERNS[lang]

    raw_counts = [Counter() for _ in range(N_SEGMENTS)]
    lemma_counts = [Counter() for _ in range(N_SEGMENTS)]
    segment_totals = [0] * N_SEGMENTS

    dataset = load_dataset('wikimedia/wikipedia', name=f'20231101.{lang}',
                           split='train', streaming=True)

    total = 0
    next_report = 0
    started = time.time()
    articles = 0

    for article in dataset:
        articles += 1
        for token in pattern.findall(article['text']):
            if total >= TARGET_TOKENS:
                break
            word = token.lower()
            segment = min(total // SEGMENT_SIZE, N_SEGMENTS - 1)

            if word in target_raw:
                raw_counts[segment][word] += 1

            lemma = simplemma.lemmatize(word, lang=lang)
            if lemma in target_lemma:
                lemma_counts[segment][lemma] += 1

            segment_totals[segment] += 1
            total += 1

            if total >= next_report:
                elapsed = time.time() - started
                print(f'{lang}: {total:,} / {TARGET_TOKENS:,} токенов, '
                      f'статей {articles:,}, {elapsed:.0f} с')
                next_report += 500_000

        if total >= TARGET_TOKENS:
            break

    payload = {
        'version': version,
        'lang': lang,
        'target_tokens': TARGET_TOKENS,
        'actual_tokens': int(total),
        'n_segments': N_SEGMENTS,
        'segment_totals': segment_totals,
        'top_n': TOP_N,
        'raw': {w: [int(raw_counts[s].get(w, 0)) for s in range(N_SEGMENTS)]
                for w in sorted(target_raw)},
        'lemma': {w: [int(lemma_counts[s].get(w, 0)) for s in range(N_SEGMENTS)]
                  for w in sorted(target_lemma)},
    }

    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False)
    print(f'{lang}: сохранено в {cache_path} ({cache_path.stat().st_size / 1024:.1f} КБ)')
    return payload

segments = {}
for lang in ['ru', 'en']:
    segments[lang] = build_segment_counts(lang)
    print('Сегменты:', segments[lang]['segment_totals'])


ru: загружен кэш segments_ru.json (токенов 5,000,000)
Сегменты: [1250000, 1250000, 1250000, 1250000]
en: загружен кэш segments_en.json (токенов 5,000,000)
Сегменты: [1250000, 1250000, 1250000, 1250000]


In [4]:
def juilland_stats(counts):
    x = np.asarray(counts, dtype=float)
    n = len(x)
    mu = float(x.mean())
    sigma = float(x.std(ddof=0))
    if mu <= 0 or n < 2:
        return np.nan, mu, sigma
    d = 1.0 - sigma / (mu * np.sqrt(n - 1))
    return float(d), mu, sigma

rows = []
for key, label, filename, lang, variant in DICTIONARIES:
    df = datasets[key]['df'].head(10)
    counts_map = segments[lang][variant]
    total_tokens = packages[key]['total_tokens']
    for _, row in df.iterrows():
        word = row['word']
        counts = counts_map.get(word, [0] * N_SEGMENTS)
        d, mu, sigma = juilland_stats(counts)
        rows.append({
            'key': key,
            'dictionary': label,
            'word': word,
            'freq': int(row['freq']),
            'rel_freq_per_1e6': row['freq'] / total_tokens * 1e6,
            'rank': int(row['rank']),
            'segments': counts,
            'mu_segment': mu,
            'sigma_segment': sigma,
            'D': d,
        })

table = pd.DataFrame(rows)
table.to_csv(DATA / 'juilland_table_top10.csv', index=False, float_format='%.8f')

display(table[['dictionary', 'word', 'rel_freq_per_1e6', 'rank', 'D']]
        .rename(columns={
            'dictionary': 'Словарь',
            'word': 'Слово',
            'rel_freq_per_1e6': 'Отн. частота (ipm)',
            'rank': 'Ранг',
            'D': 'D',
        }).round(3))


,Словарь,Слово,Отн. частота (ipm),Ранг,D
0,"Русский, без лемматизации",в,46220.491,1,0.977
1,"Русский, без лемматизации",и,34673.396,2,0.991
2,"Русский, без лемматизации",на,14288.461,3,0.970
3,"Русский, без лемматизации",с,11759.837,4,0.981
4,"Русский, без лемматизации",по,6612.274,5,0.962
5,"Русский, без лемматизации",из,5524.880,6,0.981
6,"Русский, без лемматизации",года,5106.620,7,0.947
7,"Русский, без лемматизации",к,5060.258,8,0.970
8,"Русский, без лемматизации",не,4796.337,9,0.961
9,"Русский, без лемматизации",году,4650.830,10,0.973


In [5]:
lines = ['### Вывод по заданию 3', '']
for label, group in table.groupby('dictionary', sort=False):
    lines.append(
        f"- **{label}**: D для десяти самых частых слов — от {group['D'].min():.3f} "
        f"до {group['D'].max():.3f}."
    )

lines += [
    '',
    'Для самых частых слов коэффициент D близок к 1: они встречаются во всех сегментах корпуса '
    'примерно с одинаковой относительной частотой. Это подтверждает ожидаемое поведение '
    'коэффициента Жуйана для служебной и высокочастотной лексики.'
]
display(Markdown('\n'.join(lines)))


### Вывод по заданию 3

- **Русский, без лемматизации**: D для десяти самых частых слов — от 0.947 до 0.991.
- **Русский, с лемматизацией**: D для десяти самых частых слов — от 0.960 до 0.991.
- **Английский, без лемматизации**: D для десяти самых частых слов — от 0.951 до 0.997.
- **Английский, с лемматизацией**: D для десяти самых частых слов — от 0.975 до 0.997.

Для самых частых слов коэффициент D близок к 1: они встречаются во всех сегментах корпуса примерно с одинаковой относительной частотой. Это подтверждает ожидаемое поведение коэффициента Жуйана для служебной и высокочастотной лексики.